# ParticleTracer event displays

Reads the ROOT files written by `Offline/STMMC/src/ParticleTracer_module.cc`
(fcl: `Offline/STMMC/fcl/ParticleTracer.fcl`) and draws one top/side view display per
**matched** particle: starting from the particle that reached the VD and walking back
through its genealogy tree to the primary.

One TTree entry = one `SimParticle`: either a particle that reached the seeded
StepPointMCs and passed the PDG filter (`matched == True`) or one of its ancestors up to
the primary (`matched == False`). A single art event can hold several matched particles,
each with its own chain, so the unit of a display is the matched particle rather than
the art event.

* **solid line** - the entry has a stored `MCTrajectory` (`hasTrajectory == True`); the
  line runs through all of its points.
* **dashed line** - no `MCTrajectory` was stored (the particle failed the G4 trajectory
  cuts), so the entry only holds the `SimParticle` start and end positions and the line
  is a straight segment between them.

Colour is by PDG ID (`pdgid.pdgid_color_dict`); the matched particle is drawn slightly
thicker than its ancestors.

In [ ]:
from __future__ import print_function
import sys, os
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

import ROOT
from ROOT import gROOT, gStyle, gDirectory, gPad

import filepath
import portROOT2pd_particletracer
from pdgid import pdgid_dict
import plot_utils
import constants

geometry = "MDC2025ab"
tags = ["Ele", "Mu", "1809", "Neutrals101", "Neutrals116"]
pklname = geometry + "_particletracer_neutron.pkl"

# Output of ParticleTracer.fcl. 
particle_tracer_root_files = {
    "MDC2025ab": {
        "Ele"     : ["/exp/mu2e/data/users/yongyiwu/MDC2025ab/datasets/MDC2025/rootfiles/EleBeamToVD101Neutron.root"],
        "Mu"      : ["/exp/mu2e/data/users/yongyiwu/MDC2025ab/datasets/MDC2025/rootfiles/MuBeamToVD101Neutron.root"],
        "1809"    : ["/exp/mu2e/data/users/yongyiwu/MDC2025ab/datasets/MDC2025/rootfiles/1809BeamToVD101Neutron.root"],
        "Neutrals101": ["/exp/mu2e/data/users/yongyiwu/MDC2025ab/datasets/MDC2025/rootfiles/NeutralsToVD101Neutron.root"],
        "Neutrals116": ["/exp/mu2e/data/users/yongyiwu/MDC2025ab/datasets/MDC2025/rootfiles/NeutralsToVD116Neutron.root"],
    }
}

In [ ]:
df_traj = pd.DataFrame()
for tag in tags:
    fileList_ = particle_tracer_root_files[geometry][tag]
    dft_ = portROOT2pd_particletracer.PortToDF(geometry, tag, fileList_, verbose=True,
                                               treedir="particleTracer", treename="ttree",
                                               weighted=False)
    df_traj = pd.concat([df_traj, dft_], ignore_index=True)
with open(pklname, 'wb') as f:
    pickle.dump(df_traj, f)

In [ ]:
with open(pklname, 'rb') as f:
    df_traj = pickle.load(f)
display(df_traj)

print("entries:              ", len(df_traj))
print("with MCTrajectory:    ", int(df_traj['hasTrajectory'].sum()), "(solid)")
print("start/end only:       ", int((~df_traj['hasTrajectory']).sum()), "(dashed)")
print("matched (VD) parts:   ", int(df_traj['matched'].sum()))
print("ancestors:            ", int((~df_traj['matched']).sum()))
print("pdgIds present:       ", np.sort(df_traj['pdgId'].unique()))
print("art events:           ", len(portROOT2pd_particletracer.getEventList(df_traj)))

# one row per particle that reached the VD; each gets its own display below
df_matched = portROOT2pd_particletracer.getMatched(df_traj)
print("matched particles:    ", len(df_matched))
display(df_matched[['tag', 'fileno', 'run', 'subRun', 'event', 'simId', 'pdgId',
                    'hasTrajectory', 'nPoints', 'parentPdgId', 'creationCode',
                    'startz', 'starttime', 'startkE', 'endz', 'endtime', 'endkE']])

## One display per matched particle

For each particle that reached the VD (`matched == True`), walk its genealogy tree back
to the primary and draw the whole chain: the matched particle first, then parent,
grandparent, ... , primary last (the order `ancestorSimIds` is written in).

Set `nshow`/`stride` to control how many matched particles are scanned, and use
`sel` to restrict to a tag or a PDG ID.

In [ ]:
chaincols = ['simId', 'pdgId', 'matched', 'hasTrajectory', 'nPoints',
             'parentSimId', 'parentPdgId', 'creationCode', 'isPrimary',
             'startx', 'starty', 'startz', 'starttime', 'startkE',
             'endx', 'endy', 'endz', 'endtime', 'endkE']

def chain_title(seed_, nchain):
    try:
        seed_name = pdgid_dict[seed_['pdgId']]
    except KeyError:
        seed_name = str(int(seed_['pdgId']))
    return ("Back trace of " + str(seed_['tag']) + " %03i " % seed_['fileno'] + seed_name +
            "  run %i subRun %i event %i simId %i" % (seed_['run'], seed_['subRun'],
                                                      seed_['event'], seed_['simId']) +
            "  (%i in chain)" % nchain)

# sel = df_matched.query("tag=='Ele'").reset_index(drop=True)   # restrict by tag / pdgId here
sel = df_matched

nshow = 10   # how many matched particles to draw
stride = 1   # step through the matched particle list

for ii in range(0, min(nshow*stride, len(sel)), stride):
    print('------------------------------------------------------------------------------')
    seed_ = sel.iloc[ii]
    # seed first, then parent, grandparent, ... , primary last
    dfc_ = portROOT2pd_particletracer.getGenealogy(df_traj, seed_)
    display(dfc_[chaincols])
    fig, ax_top, ax_side = plot_utils.draw_particle_tracer_event(dfc_, chain_title(seed_, len(dfc_)))
    plt.show()

## Save the displays to a PDF

In [ ]:
nsave = 20   # one page per matched particle
pdfname = geometry + "_particletracer_displays.pdf"
with PdfPages(pdfname) as pdf:
    for ii in range(min(nsave, len(sel))):
        seed_ = sel.iloc[ii]
        dfc_ = portROOT2pd_particletracer.getGenealogy(df_traj, seed_)
        fig, ax_top, ax_side = plot_utils.draw_particle_tracer_event(dfc_, chain_title(seed_, len(dfc_)))
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)
print("written " + pdfname)